# Reconstrucción auxiliar de jugadores Sub-21 (2018–2021)

Este notebook consulta dos fuentes estadísticas auxiliares: Classificação Geral, mediante REST, y Tribuna, mediante GraphQL. Estas fuentes entregan estadísticas generales de partidos y minutos; **no sustituyen las planillas contables oficiales de minutos de ANFP**.

La consulta de Classificação Geral fue probada el 24 de noviembre de 2026 y respondió correctamente. La consulta GraphQL de Tribuna queda incluida con diagnóstico porque el endpoint respondió `400: no queries to execute` durante la prueba.

In [ ]:
from pathlib import Path
import json
import time
import requests
import pandas as pd

OUT = Path('salida_notebook_minutaje')
OUT.mkdir(exist_ok=True)
CG_BASE = 'https://api.classificacaogeral.com.br/v1/competitions/265/editions'
CG_PROFILE = 'https://api.classificacaogeral.com.br/v1/players'
TRIBUNA_ENDPOINT = 'https://api-gateway.tribuna.com/v2/graphql/'
SEASONS = ['2018', '2019', '2020', '2021']
CUTOFFS = {'2018': '1998-01-01', '2019': '1998-01-01', '2020': '1999-01-01', '2021': '2000-01-01'}

## 1. Classificação Geral: descarga paginada

La ruta base es `https://api.classificacaogeral.com.br/v1/competitions/265/editions/{temporada}/players`. La respuesta observada contiene `items`, `total`, `page` y `pageSize`, por lo que el notebook avanza por páginas y conserva una copia JSON cruda por temporada.

In [ ]:
def fetch_classificacao(season, timeout=30):
    url = f'{CG_BASE}/{season}/players'
    rows = []
    page = 1
    while True:
        response = requests.get(url, params={'page': page}, timeout=timeout)
        response.raise_for_status()
        payload = response.json()
        (OUT / f'classificacao_{season}_pagina_{page}.json').write_text(
            json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        items = payload.get('items', [])
        rows.extend(items)
        total = payload.get('total')
        page_size = payload.get('pageSize') or len(items)
        if not items or (total is not None and page * page_size >= total):
            break
        page += 1
        time.sleep(0.15)
    return rows

cg_rows = []
for season in SEASONS:
    rows = fetch_classificacao(season)
    for row in rows:
        row = dict(row)
        row['season'] = season
        cg_rows.append(row)
    print(season, len(rows))

cg_raw = pd.DataFrame(cg_rows)
cg_raw.to_csv(OUT / 'classificacao_general_paginated.csv', index=False, encoding='utf-8-sig')
cg_raw.head()

In [ ]:
def fetch_profile(player_id, timeout=30):
    url = f'{CG_PROFILE}/{player_id}'
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return response.json()

# Activar solo si se necesitan fechas de nacimiento y perfiles.
RUN_PROFILES = False

if RUN_PROFILES:
    profiles = {}
    for player_id in sorted(cg_raw['playerId'].dropna().unique()):
        profiles[int(player_id)] = fetch_profile(int(player_id))
        time.sleep(0.1)
    (OUT / 'classificacao_profiles.json').write_text(
        json.dumps(profiles, ensure_ascii=False, indent=2), encoding='utf-8'
    )
else:
    profiles = {}

def eligible(row):
    nationality = str(row.get('nationality', '')).strip()
    birth = profiles.get(int(row['playerId']), {}).get('birthDate', '') if profiles else ''
    return nationality == 'Chile' and int(row.get('minutes') or 0) > 0 and (not birth or birth >= CUTOFFS[row['season']])

eligible_rows = cg_raw[cg_raw.apply(eligible, axis=1)].copy()
eligible_rows.to_csv(OUT / 'classificacao_elegibles.csv', index=False, encoding='utf-8-sig')
eligible_rows.head()

## 2. Tribuna: consulta GraphQL

Aquí la temporada no se agrega a la URL. Se envía en `variables.statSeasonID`. La petición debe ser `POST` con JSON.

In [ ]:
TRIBUNA_QUERY = '''
query getBestPlayersByMetric($statSeasonID: [String!]!, $attribute: [statRankingAttribute!]) {
  stat { football { stat_season(id: $statSeasonID) {
    rankingPlayerStat(input: { limit: 1000, attribute: $attribute }) {
      items { value rank player { id firstName lastName nationality { name }
        tag { extra { ... on TagPersonExtra { dateBirth } } } } team { name } }
      attribute
    }
  } } }
}
'''

def fetch_tribuna(season_id, limit=1000):
    body = {
        'operationName': 'getBestPlayersByMetric',
        'query': TRIBUNA_QUERY,
        'variables': {
            'statSeasonID': [season_id],
            'attribute': ['TOTAL_MINS_PLAYED'],
        },
    }
    headers = {
        'Content-Type': 'application/json',
        'X-Language': 'es',
        'Origin': 'https://tribuna.com',
        'Referer': 'https://tribuna.com/',
        'User-Agent': 'Mozilla/5.0',
    }
    response = requests.post(TRIBUNA_ENDPOINT, json=body, headers=headers, timeout=60)
    print('HTTP', response.status_code, response.text[:300])
    response.raise_for_status()
    payload = response.json()
    if payload.get('errors'):
        raise RuntimeError(payload['errors'])
    return payload

# IDs usados en la reconstrucción histórica. Ejecutar si el endpoint vuelve a aceptar la consulta.
TRIBUNA_SEASONS = {
    '2015-2016': 'primera_division_chl_15-16',
    '2016-2017': 'primera_division_chl_16-17',
    '2017': 'primera_division_chl_2017',
}

# payload_tribuna = fetch_tribuna(TRIBUNA_SEASONS['2015-2016'])
# (OUT / 'tribuna_2015_2016.json').write_text(json.dumps(payload_tribuna, ensure_ascii=False, indent=2), encoding='utf-8')

## Interpretación y límites

- Classificação Geral respondió con datos paginados (`items`, `total`, `page`, `pageSize`).
- La paginación utilizada en este notebook es `?page=N`; conviene verificar que el servidor no cambie su contrato.
- Tribuna se consulta por `POST` a `https://api-gateway.tribuna.com/v2/graphql/`; en la prueba actual respondió `400: no queries to execute`, por lo que se debe revisar el contrato vigente antes de usarlo como fuente reproducible.
- Ninguna de estas dos APIs equivale automáticamente al minutaje reglamentario de ANFP. Sus minutos deben etiquetarse como estadísticas generales y cotejarse con planillas oficiales.